# Role
あなたは .NET 8 および Windows Server (IIS) に精通したシニアソリューションアーキテクトです。

# Task
以下の要件に基づき、Windows認証をトリガーとして独自RSA鍵で署名されたJWTを発行するWeb API（Minimal API）を構築してください。

# Requirements
1. **Authentication & Authorization**:
   - `api/auth/token` エンドポイントにて `Microsoft.AspNetCore.Authentication.Negotiate` を使用し、Windows認証を行う。
   - 認証後、`C:\inetpub\wwwroot\auth_data.xml` を参照し、`WindowsID` とクエリパラメータ `appCode` が一致するユーザーが存在するかチェックする。
   - 一致しない場合は 403 Forbidden を返す。
   - `FileSystemWatcher` を使用し、`auth_data.xml` の変更をリアルタイムで検知・再読み込み（ホットリロード）するキャッシュ機構を実装すること。

2. **Key Management (Strict Constraint)**:
   - IISの証明書ストアや自己署名証明書は使用せず、`C:\AppKeys\jwt_sig_key.pem` に独自のRSA鍵をファイル永続化する。
   - 起動時、ファイルが存在しなければ `RSA.Create(2048)` で生成・保存し、存在すれば読み込んで再利用すること。

3. **JWT Issuance**:
   - 署名アルゴリズム: RS256
   - クレーム: `name` (XMLから取得), `role` (XMLから取得), `app_code`, `sub` (WindowsID), `jti` (GUID)。
   - 有効期限: 8時間

4. **JWKS Endpoint**:
   - `api/auth/jwks` を実装。現在使用中のRSA公開鍵をJWKS形式で返し、外部サービスが検証可能にすること。

# Technical Stack
- Framework: .NET 8 (Minimal API)
- Auth: Negotiate (Windows Authentication)
- JWT: System.IdentityModel.Tokens.Jwt
- OS: Windows Server 2019 (IIS 10.0)

# Output
- `Program.cs` (DI、Middleware、Endpoints)
- `RsaKeyService.cs` (鍵のライフサイクル管理)
- `UserDataSource.cs` (XMLのキャッシュと監視)
- 必要最小限の `csproj` 設定 (NuGetパッケージ参照)
- IISにおける AppPool へのフォルダ権限付与の解説

In [ ]:
シニアソリューションアーキテクトとして、ご提示いただいた要件（Windows認証と連携した独自RSA鍵によるJWT発行、ホットリロード可能なXML認証データソース、IIS環境での運用）を満たす、堅牢かつ保守性の高いアーキテクチャを提案します。

以下に実装の詳細を提示します。

### アーキテクチャ概要

1. **RsaKeyService**: シングルトンとして動作し、RSA鍵のライフサイクル（生成・保存・読み込み）とJWKSの生成を管理します。
2. **UserDataSource**: ファイルシステムの変更監視を行い、XMLデータのメモリ内キャッシュをスレッドセーフに管理します。
3. **Minimal API Endpoint**: Windows認証ミドルウェアを通過したアイデンティティを利用し、カスタムロジックで検証後、JWTを発行します。

---

### 1. Project Configuration (`.csproj`)

最小限の構成です。Windows認証とJWT機能を含めます。

```xml
<Project Sdk="Microsoft.NET.Sdk.Web">

  <PropertyGroup>
    <TargetFramework>net8.0</TargetFramework>
    <Nullable>enable</Nullable>
    <ImplicitUsings>enable</ImplicitUsings>
    <InvariantGlobalization>false</InvariantGlobalization>
  </PropertyGroup>

  <ItemGroup>
    <PackageReference Include="Microsoft.AspNetCore.Authentication.Negotiate" Version="8.0.0" />
    <PackageReference Include="System.IdentityModel.Tokens.Jwt" Version="7.3.1" />
  </ItemGroup>

</Project>

```

---

### 2. RsaKeyService.cs (鍵管理)

RSA鍵の永続化と、署名用クレデンシャルおよびJWKSの提供を行います。

```csharp
using System.Security.Cryptography;
using Microsoft.IdentityModel.Tokens;

namespace WindowsAuthJwtApi.Services;

public class RsaKeyService
{
    private const string KeyPath = @"C:\AppKeys\jwt_sig_key.pem";
    private readonly RsaSecurityKey _securityKey;
    private readonly JsonWebKey _jwk;

    public RsaKeyService()
    {
        // ディレクトリが存在しない場合は作成
        var directory = Path.GetDirectoryName(KeyPath);
        if (!Directory.Exists(directory))
        {
            Directory.CreateDirectory(directory!);
        }

        RSA rsa;

        if (File.Exists(KeyPath))
        {
            // 既存の鍵を読み込む
            var pem = File.ReadAllText(KeyPath);
            rsa = RSA.Create();
            rsa.ImportFromPem(pem);
        }
        else
        {
            // 新規作成 (2048bit)
            rsa = RSA.Create(2048);
            // プライベートキーを含むPEMとして保存 (PKCS#8)
            var pem = rsa.ExportRSAPrivateKeyPem();
            File.WriteAllText(KeyPath, pem);
        }

        // 署名用キーの作成
        _securityKey = new RsaSecurityKey(rsa)
        {
            KeyId = "custom-rsa-key-1" // キーローテーション時に識別するためIDを付与
        };

        // JWKS公開用データの作成
        var parameters = rsa.ExportParameters(false);
        _jwk = new JsonWebKey
        {
            Kty = JsonWebAlgorithmsKeyTypes.RSA,
            Use = "sig",
            Kid = _securityKey.KeyId,
            Alg = SecurityAlgorithms.RsaSha256,
            // Modulus and Exponent are Base64Url encoded by JsonWebKey internally or explicitly here
            N = Base64UrlEncoder.Encode(parameters.Modulus),
            E = Base64UrlEncoder.Encode(parameters.Exponent)
        };
    }

    public RsaSecurityKey GetSecurityKey() => _securityKey;
    
    // JWKSエンドポイント用
    public JsonWebKeySet GetJwks()
    {
        return new JsonWebKeySet { Keys = { _jwk } };
    }
}

```

---

### 3. UserDataSource.cs (XML監視とキャッシュ)

`FileSystemWatcher` を利用したホットリロード機構です。XMLファイルへのロック競合を避けるリトライロジックを含んでいます。

```csharp
using System.Collections.Concurrent;
using System.Xml.Linq;

namespace WindowsAuthJwtApi.Services;

public record UserInfo(string WindowsId, string AppCode, string Name, string Role);

public class UserDataSource : IDisposable
{
    private const string DataPath = @"C:\inetpub\wwwroot\auth_data.xml";
    private readonly FileSystemWatcher? _watcher;
    // 高速な読み取りのためにDictionaryでキャッシュ (Key: WindowsID + AppCode)
    private Dictionary<string, UserInfo> _userCache = new();
    private readonly object _lock = new();

    public UserDataSource()
    {
        LoadData();

        // 監視設定
        var directory = Path.GetDirectoryName(DataPath);
        if (Directory.Exists(directory))
        {
            _watcher = new FileSystemWatcher(directory, Path.GetFileName(DataPath))
            {
                NotifyFilter = NotifyFilters.LastWrite | NotifyFilters.FileName | NotifyFilters.Size
            };
            _watcher.Changed += OnFileChanged;
            _watcher.Created += OnFileChanged;
            _watcher.Renamed += OnFileChanged;
            _watcher.EnableRaisingEvents = true;
        }
    }

    public UserInfo? GetUser(string windowsId, string appCode)
    {
        lock (_lock)
        {
            // WindowsIDは大文字小文字を区別しない比較が一般的
            var key = CreateKey(windowsId, appCode);
            return _userCache.TryGetValue(key, out var user) ? user : null;
        }
    }

    private string CreateKey(string windowsId, string appCode) => 
        $"{windowsId.ToLowerInvariant()}:{appCode.ToLowerInvariant()}";

    private void OnFileChanged(object sender, FileSystemEventArgs e)
    {
        // 変更検知直後はファイルがロックされている可能性があるため、少し待機してからリトライ付きで読み込む
        Task.Run(async () =>
        {
            await Task.Delay(500); // デバウンス
            LoadDataWithRetry();
        });
    }

    private void LoadDataWithRetry(int retries = 3)
    {
        for (int i = 0; i < retries; i++)
        {
            try
            {
                LoadData();
                return;
            }
            catch (IOException)
            {
                Thread.Sleep(500); // 失敗したら待機
            }
            catch (Exception ex)
            {
                Console.WriteLine($"Error loading auth_data.xml: {ex.Message}");
                return;
            }
        }
    }

    private void LoadData()
    {
        if (!File.Exists(DataPath))
        {
            lock (_lock) { _userCache = new Dictionary<string, UserInfo>(); }
            return;
        }

        try
        {
            // XML構造の想定:
            // <Users>
            //   <User WindowsId="DOMAIN\User" AppCode="APP001" Name="鈴木 一郎" Role="Admin" />
            // </Users>
            var doc = XDocument.Load(DataPath);
            var newCache = new Dictionary<string, UserInfo>();

            foreach (var el in doc.Descendants("User"))
            {
                var wId = el.Attribute("WindowsId")?.Value;
                var aCode = el.Attribute("AppCode")?.Value;
                var name = el.Attribute("Name")?.Value;
                var role = el.Attribute("Role")?.Value;

                if (!string.IsNullOrEmpty(wId) && !string.IsNullOrEmpty(aCode))
                {
                    var user = new UserInfo(wId!, aCode!, name ?? "Unknown", role ?? "User");
                    newCache[CreateKey(wId!, aCode!)] = user;
                }
            }

            lock (_lock)
            {
                _userCache = newCache;
            }
            Console.WriteLine($"[UserDataSource] Loaded {_userCache.Count} users.");
        }
        catch (Exception ex)
        {
            // 解析エラー等はログに出すが、クラッシュさせない
            Console.WriteLine($"[UserDataSource] XML Parse Error: {ex.Message}");
            throw; // 初期ロード時は例外を投げて起動失敗させる意図があるならthrow、そうでなければ握りつぶす
        }
    }

    public void Dispose()
    {
        _watcher?.Dispose();
    }
}

```

---

### 4. Program.cs (Main Entry)

DIの設定、認証ミドルウェアの構成、およびエンドポイントの定義を行います。

```csharp
using Microsoft.AspNetCore.Authentication.Negotiate;
using Microsoft.AspNetCore.Mvc;
using Microsoft.IdentityModel.Tokens;
using System.IdentityModel.Tokens.Jwt;
using System.Security.Claims;
using WindowsAuthJwtApi.Services;

var builder = WebApplication.CreateBuilder(args);

// 1. サービスの登録
// Negotiate (Windows認証) の設定
builder.Services.AddAuthentication(NegotiateDefaults.AuthenticationScheme)
    .AddNegotiate();

builder.Services.AddAuthorization();

// 独自のサービスをSingleton登録
builder.Services.AddSingleton<RsaKeyService>();
builder.Services.AddSingleton<UserDataSource>();

var app = builder.Build();

// 2. ミドルウェアパイプライン
app.UseAuthentication();
app.UseAuthorization();

// 3. エンドポイント定義

// JWKS Endpoint (Anonymous Access)
app.MapGet("/api/auth/jwks", (RsaKeyService keyService) =>
{
    // JSON Web Key Set を返す
    return Results.Json(keyService.GetJwks());
})
.AllowAnonymous(); // 認証不要

// Token Issuance Endpoint (Requires Windows Auth)
app.MapGet("/api/auth/token", async (
    HttpContext context,
    [FromQuery] string appCode,
    UserDataSource userDataSource,
    RsaKeyService keyService) =>
{
    // 1. Windows認証されたユーザーIDを取得 (例: DOMAIN\User)
    // Negotiate認証が成功していれば User.Identity.Name に値が入る
    var windowsId = context.User.Identity?.Name;

    if (string.IsNullOrEmpty(windowsId))
    {
        return Results.Unauthorized();
    }

    // 2. XMLデータソースとの照合
    var userInfo = userDataSource.GetUser(windowsId, appCode);

    if (userInfo == null)
    {
        // ユーザーが存在しない、またはAppCodeが一致しない
        return Results.Forbid(); 
    }

    // 3. JWTクレームの作成
    var claims = new List<Claim>
    {
        new Claim(JwtRegisteredClaimNames.Sub, windowsId),
        new Claim(JwtRegisteredClaimNames.Jti, Guid.NewGuid().ToString()),
        new Claim("name", userInfo.Name),
        new Claim("role", userInfo.Role),
        new Claim("app_code", userInfo.AppCode)
    };

    // 4. トークンの署名と発行
    var securityKey = keyService.GetSecurityKey();
    var credentials = new SigningCredentials(securityKey, SecurityAlgorithms.RsaSha256);

    var tokenDescriptor = new SecurityTokenDescriptor
    {
        Subject = new ClaimsIdentity(claims),
        Expires = DateTime.UtcNow.AddHours(8), // 8時間有効
        SigningCredentials = credentials,
        Issuer = "MyWindowsAuthApi", // 必要に応じて設定
        Audience = appCode           // 必要に応じて設定
    };

    var tokenHandler = new JwtSecurityTokenHandler();
    var token = tokenHandler.CreateToken(tokenDescriptor);
    var tokenString = tokenHandler.WriteToken(token);

    return Results.Ok(new { access_token = tokenString, token_type = "Bearer", expires_in = 8 * 3600 });
})
.RequireAuthorization(); // Negotiate認証を必須とする

app.Run();

```

---

### 5. IIS設定とフォルダ権限設定の解説

本番環境 (Windows Server / IIS) で正常に動作させるためには、以下の設定が不可欠です。

#### A. 前提条件の確認

1. **Windows認証機能のインストール**:
* サーバーマネージャー > 役割と機能の追加 > Webサーバー (IIS) > Webサーバー > セキュリティ > **Windows認証** がインストールされていることを確認してください。



#### B. フォルダ権限の設定 (FileSystem ACL)

IISのアプリケーションプールID（デフォルトでは `IIS AppPool\<AppPoolName>`）に対して、以下の権限を付与する必要があります。

1. **`C:\AppKeys` (鍵保存場所)**
* **対象ユーザー**: アプリケーションプールID (例: `IIS AppPool\MyApiAppPool`)
* **必要な権限**:
* **読み取り (Read)**: 既存の鍵を読み込むため。
* **書き込み (Write/Modify)**: 初回起動時に鍵ファイルを生成するため。


* *セキュリティ上の注意*: このフォルダはIISのWebルート外にあるべきです。管理者とAppPool以外のアクセス権は削除してください。


2. **`C:\inetpub\wwwroot\auth_data.xml` (認証データ)**
* **対象ユーザー**: アプリケーションプールID
* **必要な権限**:
* **読み取り (Read)**: XMLを読み込むため。


* *注意*: `FileSystemWatcher` がディレクトリ自体の監視を行うため、親フォルダ (`wwwroot` または特定のサブフォルダ) に対する「フォルダーの内容の一覧表示」権限が必要になる場合があります。



#### C. IIS マネージャーでの設定

1. **認証設定**:
* 対象のサイトまたはアプリケーションを選択し、「認証」アイコンをダブルクリックします。
* **匿名認証 (Anonymous Authentication)**: **有効** (JWKSエンドポイント用)。
* **Windows認証 (Windows Authentication)**: **有効**。
* *注意*: ASP.NET Core の `Integration` モードでは、アプリ側 (`Program.cs`) で `EnableAnonymous` や `RequireAuthorization` を制御しますが、IIS側では両方を有効にして、リクエストをアプリまでパススルーさせるのが一般的です。




2. **構成エディター (Configuration Editor)** (オプション):
* 特定のパス (`system.webServer/security/authentication/windowsAuthentication`) で詳細なプロバイダ設定（NTLM/Kerberos）が必要な場合がありますが、通常はデフォルトで動作します。



#### D. アプリケーションプール設定

* **ID**: 推奨は `ApplicationPoolIdentity` ですが、ドメインリソースへのアクセスが必要な場合などはドメインユーザーを指定することもあります。その場合、上記フォルダ権限はそのユーザーに対して設定してください。
* **プロファイルの読み込み (Load User Profile)**: RSA鍵の生成などで暗号化APIがユーザープロファイルを必要とする場合があるため、`True` に設定することを推奨します。

---

### Auth Data XML のサンプル (`auth_data.xml`)

テスト用に以下のファイルを `C:\inetpub\wwwroot\auth_data.xml` として配置してください。

```xml
<Users>
  <User WindowsId="CONTOSO\Alice" AppCode="APP001" Name="Alice Smith" Role="Admin" />
  <User WindowsId="CONTOSO\Bob" AppCode="APP002" Name="Bob Jones" Role="User" />
</Users>

```